# M1.S5 - Asynchronous Session
## Part 1 - Guided HPC workflow
### One small application, one real cluster, the main ideas from Module 1

**Estimated time: 90 minutes**

This notebook is **Part 1 of the asynchronous session**.

You will use one small numerical application to connect the main ideas from Module 1:

```text
WHY HPC?
   |
HOW DID HPC EVOLVE?
   |
HOW IS A CLUSTER ORGANIZED?
   |
HOW DO WE REQUEST RESOURCES?
   |
HOW DO WE MEASURE WHAT HAPPENED?
```

The application estimates pi using numerical integration. The mathematics is simple so the focus stays on the HPC workflow.

### What you will do

| Stage | Approx. time |
|---|---:|
| Pre-flight and concept map | 10 min |
| Inspect the real SciTech environment | 15 min |
| Compile and sanity-check the application | 15 min |
| Submit, monitor and inspect a Slurm job | 20 min |
| Change one resource request and compare | 15 min |
| Interpret performance and architecture | 10 min |
| Evidence and handoff to Part 2 | 5 min |

### Important

This is a **guided learning activity**. It is not the Practice 1 submission instructions.

After finishing this notebook, continue with:

> **Part 2 - Practice 1: Individual Assignment**

The detailed tasks, required evidence, submission format and deadline for Part 2 are in **Blackboard**.

Throughout Part 1 use the same cycle:

> **PREDICT -> RUN -> OBSERVE -> EXPLAIN**

## 1 - Module 1 in one picture

Before running anything, connect the four live sessions.

### M1.S1 - Fundamentals

A workload becomes an HPC problem when ordinary resources cannot meet a practical requirement such as:

- time-to-solution;
- throughput;
- data volume;
- memory capacity;
- computational complexity.

### M1.S2 - Evolution

HPC changed because bottlenecks moved:

```text
processor speed -> power/heat -> parallelism -> memory/data movement -> heterogeneous systems
```

### M1.S3 - Architecture

Modern HPC is hierarchical:

```text
system -> node -> socket -> core
             |
             +-> memory
             +-> accelerator
             +-> interconnect to other nodes
```

### M1.S4 - Resource management

Shared resources are allocated by a scheduler:

```text
inspect -> request -> submit -> queue -> run -> inspect results
```

### Predict

For the application in this notebook, write one sentence for each:

1. **Performance goal:** what are we trying to reduce or improve?
2. **Architecture:** which hardware resource will execute the code?
3. **Scheduler:** why do we submit through Slurm?
4. **Metric:** what measurement will tell us whether the run improved?

## 2 - The application: numerical integration of pi

We estimate pi from:

```text
pi = integral from 0 to 1 of 4 / (1 + x^2) dx
```

A simple numerical approximation evaluates many points and adds their contributions.

Conceptually:

```text
for many points:
    compute x
    evaluate 4 / (1 + x*x)
    add contribution
```

This workload is useful here because:

- it has a clear correct result;
- runtime grows with the number of integration steps;
- it is CPU-oriented;
- it is easy to compile and run;
- the serial implementation lets us separate **resource allocation** from **parallel execution**.

### Predict

If the program is serial and we allocate four CPUs instead of one, will it automatically become four times faster?

Write your prediction now.

<details>
<summary><strong>Show explanation</strong></summary>

No.

Slurm can reserve four CPUs for the job, but a serial executable still performs one instruction stream unless the program itself uses parallelism.

This distinction will be tested on the real cluster later in the notebook.

</details>

In [ ]:
import os
import re
import time
import subprocess
from pathlib import Path

MEM_VARS = ["SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"]

def clean_slurm_env():
    env = os.environ.copy()
    for key in MEM_VARS:
        env.pop(key, None)
    return env

def run_command(command):
    print("$", command)
    p = subprocess.run(
        command,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )
    if p.stdout.strip():
        print(p.stdout.strip())
    if p.stderr.strip():
        print("[stderr]")
        print(p.stderr.strip())
    print("return code:", p.returncode)
    print()
    return p

def submit_slurm(script_path):
    p = subprocess.run(
        ["sbatch", "--parsable", str(script_path)],
        capture_output=True,
        text=True,
        env=clean_slurm_env()
    )
    if p.returncode != 0:
        raise RuntimeError(p.stderr.strip() or "sbatch failed")
    job_id = p.stdout.strip().split(";")[0]
    print("Submitted Slurm job:", job_id)
    return job_id

def slurm_status(job_id):
    if not job_id:
        print("No job ID available.")
        return
    run_command(
        f"squeue -j {job_id} -o '%.10i %.10T %.10M %.5D %R'"
    )

def wait_for_job(job_id, timeout=300, poll=2):
    start = time.time()
    while time.time() - start < timeout:
        p = subprocess.run(
            ["squeue", "-h", "-j", str(job_id), "-o", "%T"],
            capture_output=True, text=True
        )
        state = p.stdout.strip()
        if not state:
            print("Job", job_id, "has left squeue.")
            return True
        print("Job", job_id, "state:", state)
        time.sleep(poll)
    print("Timed out while waiting. The job may still be queued or running.")
    return False

def show_job_output(job_id, prefix):
    path = Path(f"{prefix}_{job_id}.out")
    if not path.exists():
        print("Output file not found yet:", path)
        return ""
    text = path.read_text(errors="replace")
    print("Output file:", path)
    print(text)
    return text

print("M1.S5 helpers ready.")

## 3 - Pre-flight: where are you running?

Before compiling or submitting anything, inspect the current environment.

Collect evidence for:

- host name;
- CPU topology;
- available Slurm partitions;
- available nodes and GPU resources;
- the resources allocated to this Jupyter session;
- compiler availability.

In [ ]:
run_command("hostname")
run_command("whoami")
run_command("lscpu | grep -E 'Architecture|CPU\\(s\\)|Thread|Core|Socket|NUMA|Model name' | head -n 14")

In [ ]:
if subprocess.run(
    "command -v sinfo >/dev/null 2>&1",
    shell=True, executable="/bin/bash"
).returncode == 0:
    run_command("sinfo -a -N -o '%N %P %t %c %m %G'")
else:
    print("Slurm is not available in this environment.")

In [ ]:
print("Current Jupyter allocation:")
for key in [
    "SLURM_JOB_ID",
    "SLURM_JOB_PARTITION",
    "SLURM_NODELIST",
    "SLURM_CPUS_PER_TASK",
    "SLURM_MEM_PER_CPU",
    "SLURM_MEM_PER_NODE",
]:
    print(f"{key}={os.environ.get(key, 'not set')}")

print()
run_command("gcc --version | head -n 1")
run_command("bash -lc 'type module >/dev/null 2>&1 && echo MODULE_COMMAND=AVAILABLE || echo MODULE_COMMAND=NOT_AVAILABLE'")

### Interpret the architecture

From the actual output, complete:

1. **Current host:** ...
2. **Sockets:** ...
3. **Cores per socket:** ...
4. **NUMA domains:** ...
5. **CPU partition:** ...
6. **GPU partition:** ...
7. **Current Jupyter CPUs allocated:** ...

### Important

The hardware visible on the node is not the same as the resources allocated to your current session.

If the node exposes hundreds of logical CPUs but `SLURM_CPUS_PER_TASK=2`, your current job has **2 CPUs**, not the whole node.

## 4 - Write and compile the application

The next cell creates a small C program.

It accepts the number of integration steps as an argument and prints:

- the estimated value of pi;
- the absolute error;
- elapsed time;
- the number of integration steps.

We will first perform a small sanity check directly from Jupyter.

In [ ]:
pi_source = r'''
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>

static double now_seconds(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec + ts.tv_nsec / 1e9;
}

int main(int argc, char **argv) {
    long long steps = 1000000LL;
    if (argc > 1) {
        steps = atoll(argv[1]);
    }
    if (steps <= 0) {
        fprintf(stderr, "steps must be positive\n");
        return 2;
    }

    const double width = 1.0 / (double)steps;
    double sum = 0.0;

    double t0 = now_seconds();

    for (long long i = 0; i < steps; ++i) {
        double x = (i + 0.5) * width;
        sum += 4.0 / (1.0 + x * x);
    }

    double pi_estimate = sum * width;
    double elapsed = now_seconds() - t0;
    double error = fabs(pi_estimate - acos(-1.0));

    printf("PI_ESTIMATE=%.12f\n", pi_estimate);
    printf("ABS_ERROR=%.12e\n", error);
    printf("ELAPSED_SEC=%.6f\n", elapsed);
    printf("STEPS=%lld\n", steps);

    return 0;
}
'''

Path("m1s5_integrate_pi.c").write_text(pi_source)

compile_result = run_command(
    "gcc -O3 -std=gnu11 m1s5_integrate_pi.c -lm -o m1s5_integrate_pi"
)

if compile_result.returncode != 0:
    raise RuntimeError("Compilation failed. Read the compiler output above.")

print("Compilation: PASS")

### Sanity test

A small direct run is useful before submitting to the scheduler.

We are checking correctness, not benchmarking the cluster.

In [ ]:
sanity = run_command("./m1s5_integrate_pi 1000000")

if sanity.returncode != 0:
    raise RuntimeError("Sanity run failed.")

m = re.search(r"ABS_ERROR=([0-9.eE+-]+)", sanity.stdout)
if not m:
    raise RuntimeError("Could not find ABS_ERROR in program output.")

error = float(m.group(1))
print("Correctness:", "PASS" if error < 1e-9 else "CHECK RESULT")

## 5 - Why compile before scheduling?

The workflow has two separate questions:

**Can the application run correctly?**

and

**How should cluster resources be allocated to run it?**

Compiling and performing a small sanity test before submitting saves queue time and makes failures easier to diagnose.

### Connect to Module 1

This also shows the role of the software stack:

```text
C source
   |
compiler
   |
executable
   |
Slurm allocation
   |
CPU on a compute node
```

## 6 - Create the first real Slurm job

The first scheduled run requests:

- CPU partition;
- 1 node;
- 1 task;
- 1 CPU;
- 1 GB RAM;
- 3 minutes maximum.

The benchmark problem uses **50 million integration steps**.

A short sleep is included after the computation so you have time to observe the job in the queue. The sleep is not included in `ELAPSED_SEC`.

In [ ]:
STEPS = 50_000_000

job1_script = f'''#!/bin/bash
#SBATCH --job-name=m1s5_pi_1cpu
#SBATCH --partition=cpu
#SBATCH --output=m1s5_pi1_%j.out
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=1G
#SBATCH --time=00:03:00

set -e
cd "$SLURM_SUBMIT_DIR"

echo "=== JOB PASSPORT ==="
echo "JOB_ID=$SLURM_JOB_ID"
echo "NODE=$(hostname)"
echo "PARTITION=$SLURM_JOB_PARTITION"
echo "NODES=$SLURM_JOB_NUM_NODES"
echo "TASKS=$SLURM_NTASKS"
echo "CPUS_PER_TASK=$SLURM_CPUS_PER_TASK"

./m1s5_integrate_pi {STEPS}

sleep 8
'''

Path("m1s5_pi1.sbatch").write_text(job1_script)
print(Path("m1s5_pi1.sbatch").read_text())

### Predict before submission

Write down:

1. Which node do you think Slurm will choose?
2. Will the job start immediately or spend time PENDING?
3. What will `CPUS_PER_TASK` report?
4. Will this job use a GPU?

In [ ]:
M1S5_JOB1 = submit_slurm("m1s5_pi1.sbatch")
slurm_status(M1S5_JOB1)

## 7 - Monitor the job

Run the next cell while the job is pending or running.

If the job is PENDING, inspect `NODELIST(REASON)`.

A pending job is not necessarily a failed job. It may simply be waiting for an appropriate scheduling opportunity.

In [ ]:
slurm_status(globals().get("M1S5_JOB1"))

## 8 - Read the result and audit the job

Wait for the job to leave the queue, then inspect both:

- program output;
- Slurm accounting information.

In [ ]:
wait_for_job(globals().get("M1S5_JOB1"), timeout=300)
OUT1 = show_job_output(globals().get("M1S5_JOB1"), "m1s5_pi1")

In [ ]:
job1 = globals().get("M1S5_JOB1")
if job1:
    run_command(
        f"sacct -j {job1} "
        "--format=JobID,JobName%20,State,Elapsed,AllocCPUS,ReqMem,ExitCode "
        "-n -P"
    )
else:
    print("Run the first job submission cell first.")

### Observe

Record:

- node selected;
- CPUs per task requested;
- requested memory;
- Slurm state;
- application runtime;
- pi error.

### A small SciTech detail

Do not assume that `SLURM_CPUS_PER_TASK` and `sacct AllocCPUS` must always show the same number.

On this cluster you may see, for example:

```text
SLURM_CPUS_PER_TASK=1
sacct AllocCPUS=2
```

The node exposes **2 hardware threads per physical core**, and Slurm accounting can reflect the scheduler's allocation granularity rather than only the value written in `--cpus-per-task`.

For this exercise, use the requested `--cpus-per-task` values when comparing the **1 vs 4 CPU request**, and use `sacct` as evidence of what Slurm actually allocated and recorded.

Neither field proves that the application used all of those CPUs.

### Explain

Which information comes from the **application**, and which comes from **Slurm**?

<details>
<summary><strong>Show explanation</strong></summary>

The application reports values such as:

- numerical result;
- error;
- measured compute time.

Slurm reports values such as:

- job ID;
- node placement;
- requested/allocated resources;
- job state;
- wall-clock job duration;
- exit status.

You need both kinds of evidence to understand an HPC run.

</details>

## 9 - Change one resource request

Now change exactly one important resource:

```text
1 CPU -> 4 CPUs
```

Everything else stays the same:

- same executable;
- same number of integration steps;
- same node count;
- same memory;
- same serial algorithm.

### Predict

Will the application runtime become approximately four times faster?

Explain your answer before running the job.

In [ ]:
job4_script = f'''#!/bin/bash
#SBATCH --job-name=m1s5_pi_4cpu
#SBATCH --partition=cpu
#SBATCH --output=m1s5_pi4_%j.out
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=1G
#SBATCH --time=00:03:00

set -e
cd "$SLURM_SUBMIT_DIR"

echo "=== JOB PASSPORT ==="
echo "JOB_ID=$SLURM_JOB_ID"
echo "NODE=$(hostname)"
echo "PARTITION=$SLURM_JOB_PARTITION"
echo "NODES=$SLURM_JOB_NUM_NODES"
echo "TASKS=$SLURM_NTASKS"
echo "CPUS_PER_TASK=$SLURM_CPUS_PER_TASK"

./m1s5_integrate_pi {STEPS}
'''

Path("m1s5_pi4.sbatch").write_text(job4_script)
print(Path("m1s5_pi4.sbatch").read_text())

In [ ]:
M1S5_JOB4 = submit_slurm("m1s5_pi4.sbatch")
wait_for_job(M1S5_JOB4, timeout=300)
OUT4 = show_job_output(M1S5_JOB4, "m1s5_pi4")

## 10 - Compare the two runs

We now have a controlled comparison:

```text
same problem
same executable
same algorithm
different CPU allocation
```

That is much better evidence than comparing unrelated runs.

In [ ]:
def extract_value(text, key):
    if not text:
        return None
    m = re.search(rf"^{re.escape(key)}=([0-9.eE+-]+)$", text, re.MULTILINE)
    return float(m.group(1)) if m else None

t1 = extract_value(OUT1, "ELAPSED_SEC")
t4 = extract_value(OUT4, "ELAPSED_SEC")
pi1 = extract_value(OUT1, "PI_ESTIMATE")
pi4 = extract_value(OUT4, "PI_ESTIMATE")

print(f"{'Run':<18} {'CPUs/task requested':>20} {'Time (s)':>12}")
print("-" * 53)
print(f"{'serial job A':<18} {1:>20} {t1 if t1 is not None else float('nan'):>12.6f}")
print(f"{'serial job B':<18} {4:>20} {t4 if t4 is not None else float('nan'):>12.6f}")

if t1 and t4:
    speedup = t1 / t4
    allocation_efficiency = speedup / 4
    print()
    print(f"Apparent speedup: {speedup:.3f}x")
    print(f"Efficiency relative to 4 requested CPUs: {allocation_efficiency:.1%}")

if pi1 is not None and pi4 is not None:
    print("Same numerical result:", abs(pi1 - pi4) < 1e-12)

### Explain

Requesting 4 CPUs per task should not produce a meaningful 4x speedup because the program is still serial.

Small timing differences are normal on a shared system.

The important lesson is:

> **The scheduler can reserve parallel resources. The application must contain parallel work to use them.**

This connects directly to the next module, where you will learn how software actually expresses parallel execution.

## 11 - Speedup and efficiency

For a fixed problem:

```text
speedup = T1 / Tp

efficiency = speedup / p
```

If four processors produced a perfect 4x speedup:

```text
efficiency = 4 / 4 = 100%
```

If four processors produce only 2x speedup:

```text
efficiency = 2 / 4 = 50%
```

### Why can efficiency fall?

Possible reasons include:

- serial work;
- communication;
- synchronization;
- load imbalance;
- memory bandwidth;
- data movement;
- parallel-management overhead;
- simply allocating resources that the program does not use.

## 12 - Amdahl's Law as a prediction tool

Suppose a future parallel version of this application has a **10% serial fraction**.

Amdahl's Law predicts:

```text
S(p) = 1 / [s + (1-s)/p]
```

What happens as we keep adding processors?

In [ ]:
def amdahl_speedup(serial_fraction, processors):
    return 1.0 / (
        serial_fraction + (1.0 - serial_fraction) / processors
    )

serial_fraction = 0.10

print(f"{'Processors':>12} {'Speedup':>12} {'Efficiency':>12}")
for p in [1, 2, 4, 8, 16, 64]:
    s = amdahl_speedup(serial_fraction, p)
    e = s / p
    print(f"{p:12d} {s:12.3f} {e:12.2%}")

print()
print("Theoretical maximum speedup:", 1 / serial_fraction, "x")

### Explain

More processors continue to help, but the benefit becomes smaller.

With a 10% serial fraction, even infinitely many processors cannot exceed a theoretical **10x** speedup.

This is one reason HPC evolved toward:

- better parallel algorithms;
- faster memory systems;
- better interconnects;
- accelerators;
- workload-specific optimization.

More hardware alone is not enough.

## 13 - Read the run through the architecture

Your application did not execute in an abstract "cloud".

It followed a real path:

```text
Jupyter session
      |
Slurm scheduler
      |
CPU partition
      |
compute node
      |
CPU core(s)
      |
DRAM
```

### Architecture questions

Use your actual outputs.

1. Which **compute node** ran the job?
2. Was the workload CPU or GPU based?
3. Was the memory model inside the job shared or distributed?
4. Did the job need an inter-node network?
5. Which component would become important if the data no longer fit in one node's memory?

<details>
<summary><strong>Show explanation</strong></summary>

For this run:

- the scheduler chooses one CPU compute node;
- the program runs on CPU;
- memory is local/shared inside that one node;
- the job does not need inter-node communication because only one node was requested;
- if the problem required multiple nodes, distributed memory and the interconnect would become important.

</details>

## 14 - The evolution question

Why did HPC systems evolve toward multicore CPUs, clusters and accelerators instead of relying only on faster clock speeds?

Connect your answer to at least two of:

- power and heat;
- memory bandwidth;
- parallelism;
- data movement;
- specialization.

<details>
<summary><strong>Show explanation</strong></summary>

Increasing clock frequency alone ran into power and thermal limits.

At the same time, memory and data movement became increasingly important.

Modern systems therefore gain performance through combinations of:

- many CPU cores;
- many nodes;
- GPUs and other accelerators;
- wider/faster memory;
- high-speed interconnects.

The architecture changed because the bottleneck changed.

</details>

## 15 - Choose the right metric

For each question, choose the most useful metric.

### A

"How long did my application take to finish?"

### B

"How much faster is the 4-worker version than the 1-worker version?"

### C

"How effectively are four workers being used?"

### D

"How much data can memory deliver per second?"

### E

"How much floating-point work can the system perform per second?"

<details>
<summary><strong>Show explanation</strong></summary>

- **A:** runtime / time-to-solution
- **B:** speedup
- **C:** parallel efficiency
- **D:** memory bandwidth
- **E:** FLOPS

The right metric depends on the performance question.

</details>

## 16 - Evidence checklist

Before leaving Part 1, make sure you can point to evidence for each item:

- [ ] cluster host and architecture;
- [ ] visible Slurm partitions;
- [ ] current Jupyter allocation;
- [ ] successful compilation;
- [ ] correct sanity run;
- [ ] first Slurm job ID;
- [ ] first job output;
- [ ] `sacct` accounting evidence;
- [ ] second job with a different CPU allocation;
- [ ] runtime comparison;
- [ ] explanation of why more allocated CPUs did or did not help.

### Optional cleanup

After you have exported or saved the notebook, the next cell removes only the temporary files created by this guided lab.

In [ ]:
generated = [
    "m1s5_integrate_pi.c",
    "m1s5_integrate_pi",
    "m1s5_pi1.sbatch",
    "m1s5_pi4.sbatch",
]

for path in generated:
    p = Path(path)
    if p.exists():
        p.unlink()
        print("Removed:", path)

print()
print("Slurm output files were kept as evidence.")

## 17 - Part 1 complete

You have completed a full guided HPC workflow:

```text
UNDERSTAND THE WORKLOAD
        |
INSPECT THE SYSTEM
        |
COMPILE
        |
SANITY CHECK
        |
REQUEST RESOURCES
        |
SUBMIT
        |
MONITOR
        |
AUDIT
        |
CHANGE ONE VARIABLE
        |
COMPARE
        |
EXPLAIN
```

You have also connected that workflow to the four main ideas from Module 1:

- why HPC is needed;
- why HPC architecture evolved;
- how a cluster is organized;
- how resources and performance are managed.

# Next: Part 2 - Practice 1

Continue with **Practice 1 in Blackboard**.

Part 2 is the individual assignment. It asks you to apply the ideas more independently.

Use Blackboard as the definitive source for:

- the workload;
- laptop and cluster instructions;
- required measurements;
- required evidence;
- submission format;
- deadline.

Do not use this notebook as the source of Part 2 submission requirements.